# Week 4 Day 5 - Exercise: making the Sidekick my own

Exercise only. It uses the Sidekick from the lab, with my changes in `sidekick.py` (approval gate on the filesystem write tools) and `sidekick_tools.py` (the `days_until` and `split_cost` tools).

In [ ]:
from dotenv import load_dotenv

from sidekick import Sidekick

load_dotenv(override=True)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make the Sidekick truly yours. Start with a personal errand: try the Expedia version of the flight task, asking the Sidekick to sign in to your Expedia account first. When it reaches the login page it will call <code>request_human_help</code> and pause while you log in to its browser window, then press on with the search. Travel sites defend themselves against automation, so expect the occasional block wall; that is part of working with real-world agents. Then go further: gate the filesystem write tools behind the approval middleware in <code>sidekick.py</code>, add a tool of your own, and set the Sidekick a task you actually need done this week, with a clear success criterion.
            </span>
        </td>
    </tr>
</table>

## My solution

### Part 1: the Expedia errand

The worker's system prompt only *suggests* Google Flights, so naming Expedia in the task is enough to override it. I never give the Sidekick my password: it opens Expedia, reaches the login page, calls `request_human_help` (which pauses the run through the human-in-the-loop middleware), and I sign in myself in its browser window. Then I resume and it carries on with the search.

In [ ]:
# A fresh Sidekick, so this errand has its own plan and its own browser session
sidekick = Sidekick()
await sidekick.setup()

# The login is deliberately left to me: the task tells the agent to ask for help rather than to type credentials
expedia_task = """Go to expedia.com and sign in to my account. When you reach the login page, use request_human_help
so I can log in myself in your browser window. Once I have, find the best round-trip flight from Lima to London,
leaving about a month from now and returning a week later. I care about price first, then total journey time,
and I would rather avoid itineraries with two or more stops. Write the top three options to expedia_flights.md."""

# The criteria demand evidence from Expedia itself, so the evaluator cannot be satisfied by a Google Flights fallback
expedia_criteria = "expedia_flights.md is written with three specific options from Expedia including airline, times and price, plus a clear recommendation."

history = await sidekick.run_turn(expedia_task, expedia_criteria, history=[])
print(history[-1]["content"])

The run should now be paused on `request_human_help`. Log in to Expedia in the Sidekick's browser window, then run the next cell to approve and resume. Travel sites fight automation, so if it hits a block wall, that is part of working with real-world agents.

In [ ]:
# Approving here means: "I have logged in, carry on"
if sidekick.paused:
    history = await sidekick.resume(history)
for entry in history[-2:]:
    print(f"[{entry['role']}] {entry['content']}\n")

### Part 2: gating the filesystem write tools

In `sidekick.py` I added a `WRITE_TOOLS` list (`write_file`, `edit_file`, `move_file`, `create_directory`) and merged it into the `interrupt_on` dict of `HumanInTheLoopMiddleware`. Read-only tools are not listed, so they still run freely. The code is read when the module is imported, so I restart the kernel (or reload the module) before building a new Sidekick.

In [ ]:
import importlib
import sidekick as sidekick_module

# Pick up the edited sidekick.py without restarting the whole kernel
importlib.reload(sidekick_module)

# Close the previous Sidekick (and its browser) before creating a new one
sidekick.cleanup()
sidekick = sidekick_module.Sidekick()
await sidekick.setup()

# A harmless write task: it should now pause BEFORE the file is created
history = await sidekick.run_turn(
    message="Write a file called approval_test.md containing one sentence about why human approval matters for AI agents.",
    success_criteria="approval_test.md exists in the sandbox and contains one sentence.",
    history=[],
)
print(history[-1]["content"])
print("Paused:", sidekick.paused)

The pause message shows the exact `write_file` call, with the path and the content. I check that the file does not exist yet, and then approve.

In [ ]:
import os
print("Exists before approval:", os.path.exists("sandbox/approval_test.md"))

if sidekick.paused:
    history = await sidekick.resume(history)
print("Exists after approval:", os.path.exists("sandbox/approval_test.md"))
for entry in history[-2:]:
    print(f"[{entry['role']}] {entry['content']}\n")

### Part 3: a tool of my own

Language models are unreliable at date and money arithmetic, so I gave the Sidekick two small deterministic tools in `sidekick_tools.py`: `days_until(date)` and `split_cost(total, people, tip_percent)`. The docstring is what the model reads to decide when to use each one, so it says exactly that. Both are registered in `get_all_tools`. First I call them directly, with no model involved, to prove they work.

In [ ]:
import importlib
import sidekick_tools
import sidekick as sidekick_module

# sidekick_tools first, because sidekick.py imports get_all_tools from it
importlib.reload(sidekick_tools)
importlib.reload(sidekick_module)

# @tool objects are called with .invoke and a dict of arguments
print(sidekick_tools.days_until.invoke({"date": "2026-12-25"}))
print(sidekick_tools.split_cost.invoke({"total": 183.40, "people": 4, "tip_percent": 15}))

In [ ]:
sidekick.cleanup()
sidekick = sidekick_module.Sidekick()
await sidekick.setup()
print([tool.name for tool in sidekick.tools if tool.name in ("days_until", "split_cost")])

history = await sidekick.run_turn(
    message="How many days until Christmas Day 2026, and if a dinner costs 183.40 split between 4 people with a 15% tip, what does each person pay?",
    success_criteria="The reply gives the number of days and the per-person amount, and the days_until and split_cost tools were both called.",
    history=[],
)
print(history[-2]["content"])
print(history[-1]["content"])

### Part 4: a real task, a LinkedIn content calendar

This week I learned how to build agents with LangChain and LangGraph, from the building blocks to the Sidekick. I want to share that on LinkedIn, so I give the Sidekick the job of planning the posts. It uses most of what I built in Parts 1 to 3:

- **Web search** to find a real, current angle or reference for each post.
- **`days_until`** to put true calendar dates on the plan instead of guessing weekdays.
- **The approval gate** when it writes the file, because `write_file` now pauses for me.
- **A push notification** (also gated) to tell me the calendar is ready.

The success criteria are concrete on purpose, so the evaluator can check them against the tool calls: five dated posts, a full draft for each, a length limit and a source link.

#### Second run: fixing what the first run missed

The first calendar met every criterion I wrote, but it had four weaknesses I found by reading it: sources were reused across days (and one did not match its topic), the drafts were generic, the links were documentation landing pages rather than current references, and nobody had checked that the links load. Each weakness becomes a sharper instruction in the task and a checkable line in the criteria:

- **Different sources:** every URL must be unique and match the day's topic.
- **Opened in the browser:** the Sidekick must visit each link, so the evaluator can see `browser_navigate` calls as evidence that the page loads.
- **My own work:** each draft must mention one concrete thing I actually built or saw this week, which I list in the task, so the model does not invent experiences.
- **More recent:** prefer articles or announcements over documentation home pages.

The result goes to `linkedin_calendar.md`, so I can compare both versions.

In [ ]:
sidekick.cleanup()
sidekick = sidekick_module.Sidekick()
await sidekick.setup()

linkedin_task = """Create a LinkedIn content calendar for the next 5 weekdays, starting tomorrow, about what I learned this week
in an AI engineering course on building agents. One topic per day, in this order, each with the real thing I did that I want
the post to mention (do not invent other experiences):
1. The building blocks of an agent: LLM, tools, memory. I started with a simple agent with search, Wikipedia and a push notification tool.
2. LangGraph: orchestration with state, nodes and edges. I drew the graph of my agent as a diagram and read its nodes and edges.
3. create_agent and middleware: planning, guardrails, human approval. I made my agent pause for my approval before it could write a file.
4. MCP servers: a real browser and a filesystem. I asked my agent to search flights on Expedia; it paused so I could log in myself,
   and I noticed it searched from Lima instead of New York because the site prefilled the origin.
5. The Sidekick: a worker with an evaluator loop. I used it to plan this very calendar, and my first run reused sources and sounded generic.
For each day, search the web for one article, announcement or tutorial published in 2025 or 2026 (not a documentation home page)
that matches the topic. Every source URL must be different. Open each URL in the browser to confirm it loads before you use it.
Use days_until to get the exact date of each post.
Write the calendar to linkedin_calendar.md. Each entry needs the date and weekday, the topic, a hook line, a full draft post of
at most 1,000 characters in the first person that mentions the thing I did and ends with a practical takeaway, three hashtags,
and the source link. Finish by sending me a push notification saying the calendar is ready."""

linkedin_criteria = """linkedin_calendar.md is written with exactly 5 entries on consecutive weekdays starting tomorrow,
each with a date and weekday, a topic from the list, a hook, a draft of at most 1,000 characters that mentions the specific thing
the user did for that day, three hashtags and a source URL. All 5 source URLs are different and each was opened with browser_navigate.
The days_until tool was called, and a push notification was sent."""

history = await sidekick.run_turn(linkedin_task, linkedin_criteria, history=[])
print(history[-1]["content"])

The Sidekick pauses at every gated action: the `write_file` and then the push notification. The cell below shows each pending action and asks me before approving. If I answer anything other than `y`, it stops and the Sidekick stays paused.

In [ ]:
# Approve each pause one at a time, reading what the Sidekick wants to do first
while sidekick.paused:
    print(history[-1]["content"])
    if input("Approve? (y/n) ").strip().lower() != "y":
        break
    history = await sidekick.resume(history)

for entry in history[-2:]:
    print(f"[{entry['role']}] {entry['content']}\n")

In [ ]:
# The deliverable, and the plan the Sidekick followed to get there
print(open("sandbox/linkedin_calendar.md").read())
print()
for todo in sidekick.todos:
    print(f"[{todo['status']}] {todo['content']}")